In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# describe(), 결측치 표, 상관계수 표를 보기 좋게 만들기 위한 기본 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [2]:
music_df = pd.read_csv('../data/train.csv')
census_df = pd.read_csv('../data/acs2015_census_tract_data.csv')

In [3]:
# location 오타 수정
music_df['location'] = music_df['location'].replace({
    'Nebrasksa': 'Nebraska'
})

In [4]:
# 원본 데이터 기본 점검
music_raw = music_df.copy()
census_raw = census_df.copy()

print(f"music_df shape: {music_df.shape}")
print(f"census_df shape: {census_df.shape}")

raw_summary_music = pd.DataFrame({
    'dtype': music_df.dtypes.astype(str),
    'missing_cnt': music_df.isna().sum(),
    'missing_ratio(%)': (music_df.isna().mean() * 100).round(2),
    'nunique': music_df.nunique(dropna=False)
}).sort_values(['missing_cnt', 'nunique'], ascending=[False, False])

display(raw_summary_music)

num_cols_music = music_df.select_dtypes(include=np.number).columns.tolist()
cat_cols_music = music_df.select_dtypes(exclude=np.number).columns.tolist()

print("수치형 컬럼:", num_cols_music)
print("범주형 컬럼:", cat_cols_music)

display(music_df[num_cols_music].describe().T)

print("\n[churned 분포]")
display(
    music_df['churned']
    .value_counts(dropna=False)
    .rename_axis('churned')
    .reset_index(name='count')
    .assign(ratio=lambda x: (x['count'] / len(music_df) * 100).round(2))
)

eda_cat_cols = ['location', 'subscription_type', 'payment_plan', 'payment_method', 'customer_service_inquiries']
for col in eda_cat_cols:
    print(f"\n[{col}]")
    display(
        music_df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name='count')
        .assign(ratio=lambda x: (x['count'] / len(music_df) * 100).round(2))
        .head(20)
    )

music_df shape: (125000, 20)
census_df shape: (74001, 37)


,dtype,missing_cnt,missing_ratio(%),nunique
customer_id,int64,0,0.0000,125000
weekly_hours,float64,0,0.0000,125000
average_session_length,float64,0,0.0000,124996
song_skip_rate,float64,0,0.0000,124995
signup_date,int64,0,0.0000,2922
weekly_songs_played,int64,0,0.0000,497
weekly_unique_songs,int64,0,0.0000,297
num_platform_friends,int64,0,0.0000,200
num_playlists_created,int64,0,0.0000,100
age,int64,0,0.0000,62


수치형 컬럼: ['customer_id', 'age', 'num_subscription_pauses', 'signup_date', 'weekly_hours', 'average_session_length', 'song_skip_rate', 'weekly_songs_played', 'weekly_unique_songs', 'num_favorite_artists', 'num_platform_friends', 'num_playlists_created', 'num_shared_playlists', 'notifications_clicked', 'churned']
범주형 컬럼: ['location', 'subscription_type', 'payment_plan', 'payment_method', 'customer_service_inquiries']


,count,mean,std,min,25%,50%,75%,max
customer_id,"125,000.0000","62,500.5000","36,084.5362",1.0000,"31,250.7500","62,500.5000","93,750.2500","125,000.0000"
age,"125,000.0000",48.4141,17.9010,18.0000,33.0000,48.0000,64.0000,79.0000
num_subscription_pauses,"125,000.0000",1.9911,1.4172,0.0000,1.0000,2.0000,3.0000,4.0000
signup_date,"125,000.0000","-1,460.6789",844.1329,"-2,922.0000","-2,190.0000","-1,462.0000",-728.0000,-1.0000
weekly_hours,"125,000.0000",25.0370,14.4475,0.0001,12.4727,25.1167,37.5703,49.9999
average_session_length,"125,000.0000",60.4217,34.3838,1.0005,30.6442,60.3410,90.2342,119.9965
song_skip_rate,"125,000.0000",0.5008,0.2887,0.0000,0.2510,0.5012,0.7511,1.0000
weekly_songs_played,"125,000.0000",250.8239,143.3276,3.0000,127.0000,251.0000,375.0000,499.0000
weekly_unique_songs,"125,000.0000",150.7833,85.7950,3.0000,76.0000,150.0000,225.0000,299.0000
num_favorite_artists,"125,000.0000",24.4999,14.4460,0.0000,12.0000,25.0000,37.0000,49.0000



[churned 분포]


...

,payment_plan,count,ratio
0,Monthly,62562,50.0500
1,Yearly,62438,49.9500



[payment_method]


,payment_method,count,ratio
0,Debit Card,31292,25.0300
1,Paypal,31282,25.0300
2,Credit Card,31213,24.9700
3,Apple Pay,31213,24.9700



[customer_service_inquiries]


,customer_service_inquiries,count,ratio
0,Low,41873,33.5000
1,High,41583,33.2700
2,Medium,41544,33.2400


In [5]:
# customer_id 삭제
if 'customer_id' in music_df.columns:
    music_df = music_df.drop('customer_id', axis=1)

In [6]:
income_missing_before = census_df['Income'].isna().sum()
zero_totalpop_cnt = (census_df['TotalPop'] == 0).sum()

# Census 데이터 전처리
census_df['Income'] = census_df.groupby('State')['Income'].transform(lambda x: x.fillna(x.median()))

# State별 집계
state_stats = census_df.groupby('State').agg({
    'TotalPop': 'sum',
    'Income': 'mean'
}).reset_index()

# 컬럼명 변경
state_stats.columns = ['State', 'State_TotalPop', 'State_AvgIncome']

print(f"Income 결측치(전): {income_missing_before}")
print(f"Income 결측치(후): {census_df['Income'].isna().sum()}")
print(f"TotalPop == 0 개수: {zero_totalpop_cnt}")

print("\n[state_stats 기본 통계]")
display(state_stats.describe().T)

music_states = set(music_df['location'].dropna().unique())
census_states = set(state_stats['State'].dropna().unique())

only_music = sorted(music_states - census_states)
only_census = sorted(census_states - music_states)

print("\nmusic_df에만 있는 location 값")
print(only_music)

print("\nstate_stats에만 있는 State 값")
print(only_census)

Income 결측치(전): 1100
Income 결측치(후): 0
TotalPop == 0 개수: 690

[state_stats 기본 통계]


,count,mean,std,min,25%,50%,75%,max
State_TotalPop,52.0000,"6,155,732.5769","6,992,998.8097","579,679.0000","1,792,701.7500","4,168,293.0000","6,775,555.5000","38,421,464.0000"
State_AvgIncome,52.0000,"55,919.9565","11,361.0298","20,513.9058","48,477.0198","54,188.0266","63,634.0427","78,657.1771"



music_df에만 있는 location 값
[]

state_stats에만 있는 State 값
['Alaska', 'Arizona', 'Arkansas', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Hawaii', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Nevada', 'New Hampshire', 'New Mexico', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Puerto Rico', 'Rhode Island', 'South Dakota', 'Tennessee', 'Texas', 'Wyoming']


In [7]:
# 데이터 병합 + 병합 상태 점검
final_df = pd.merge(
    music_df,
    state_stats,
    left_on='location',
    right_on='State',
    how='left',
    indicator=True
)

print("병합 결과 요약")
display(
    final_df['_merge']
    .value_counts(dropna=False)
    .rename_axis('_merge')
    .reset_index(name='count')
    .assign(ratio=lambda x: (x['count'] / len(final_df) * 100).round(2))
)

print("\n매칭 실패 location 상위")
display(
    final_df.loc[final_df['_merge'] != 'both', 'location']
    .value_counts(dropna=False)
    .rename_axis('location')
    .reset_index(name='count')
    .head(20)
)

병합 결과 요약


,_merge,count,ratio
0,both,125000,100.0000
1,left_only,0,0.0000
2,right_only,0,0.0000



매칭 실패 location 상위


,location,count


In [8]:
# 병합 확인용 컬럼 제거
final_df = final_df.drop(['State', '_merge'], axis=1)

In [9]:
# 수치 데이터 보정
final_df['average_session_length'] = final_df['average_session_length'] / 60
final_df['weekly_unique_songs'] = np.where(
    final_df['weekly_unique_songs'] > final_df['weekly_songs_played'],
    final_df['weekly_songs_played'],
    final_df['weekly_unique_songs']
)

final_df['tenure_days'] = final_df['signup_date'].abs()
final_df = final_df.drop('signup_date', axis=1)

In [10]:
# 전처리 논리 점검
logic_check = pd.Series({
    'weekly_unique_songs > weekly_songs_played': int((final_df['weekly_unique_songs'] > final_df['weekly_songs_played']).sum()),
    'average_session_length <= 0': int((final_df['average_session_length'] <= 0).sum()),
    'song_skip_rate < 0 or > 1': int(((final_df['song_skip_rate'] < 0) | (final_df['song_skip_rate'] > 1)).sum()),
    'weekly_hours < 0': int((final_df['weekly_hours'] < 0).sum()),
    'tenure_days < 0': int((final_df['tenure_days'] < 0).sum())
}, name='count')

display(logic_check.to_frame())

print("\ntenure_days 요약")
display(final_df['tenure_days'].describe().to_frame().T)

# 가설 EDA용 원본 형태 백업
eda_df = final_df.copy()

,count
weekly_unique_songs > weekly_songs_played,0
average_session_length <= 0,0
song_skip_rate < 0 or > 1,0
weekly_hours < 0,0
tenure_days < 0,0



tenure_days 요약


,count,mean,std,min,25%,50%,75%,max
tenure_days,"125,000.0000","1,460.6789",844.1329,1.0000,728.0000,"1,462.0000","2,190.0000","2,922.0000"


In [11]:
# 문자열 처리
final_df = final_df.drop('location', axis=1, errors='ignore')

cols_to_encode = ['subscription_type', 'payment_plan', 'payment_method', 'customer_service_inquiries']
final_df = pd.get_dummies(final_df, columns=cols_to_encode, drop_first=True, dtype=int)

In [12]:
# 최종 데이터 결과 확인
print(f"최종 데이터 형태: {final_df.shape}")
display(final_df.head())

print("\n최종 데이터 기본 통계")
display(final_df.describe().T)

print("\n최종 데이터 결측치 요약")
missing_summary = pd.DataFrame({
    'missing_cnt': final_df.isna().sum(),
    'missing_ratio(%)': (final_df.isna().mean() * 100).round(2),
    'nunique': final_df.nunique(dropna=False)
}).sort_values(['missing_cnt', 'nunique'], ascending=[False, False])

display(missing_summary)

print("\n중복 행 수:", final_df.duplicated().sum())

print("\n타깃(churned) 분포")
display(
    final_df['churned']
    .value_counts(dropna=False)
    .rename_axis('churned')
    .reset_index(name='count')
    .assign(ratio=lambda x: (x['count'] / len(final_df) * 100).round(2))
)

constant_cols = [col for col in final_df.columns if final_df[col].nunique(dropna=False) <= 1]
print("\n상수 컬럼:", constant_cols)

최종 데이터 형태: (125000, 25)


,age,num_subscription_pauses,weekly_hours,average_session_length,song_skip_rate,weekly_songs_played,weekly_unique_songs,num_favorite_artists,num_platform_friends,num_playlists_created,num_shared_playlists,notifications_clicked,churned,State_TotalPop,State_AvgIncome,tenure_days,subscription_type_Free,subscription_type_Premium,subscription_type_Student,payment_plan_Yearly,payment_method_Credit Card,payment_method_Debit Card,payment_method_Paypal,customer_service_inquiries_Low,customer_service_inquiries_Medium
0,32,2,22.3914,1.7566,0.1769,169,109,18,32,52,35,46,0,1014699,"47,631.1568",1606,1,0,0,1,0,0,1,0,1
1,64,3,29.2942,0.8750,0.9818,55,55,44,33,12,25,37,1,8904413,"76,533.2965",2897,1,0,0,0,0,0,1,1,0
2,51,2,15.4003,0.4117,0.0484,244,117,20,129,50,28,38,0,6985464,"64,452.0583",348,0,1,0,1,1,0,0,0,0
3,63,4,22.8421,1.3933,0.0357,442,252,47,120,55,17,24,0,38421464,"67,180.1817",2894,0,0,0,1,0,0,0,0,1
4,54,3,23.1512,0.8763,0.0397,243,230,41,66,40,32,47,0,6985464,"64,452.0583",92,0,0,0,0,0,0,1,0,0



최종 데이터 기본 통계


,count,mean,std,min,25%,50%,75%,max
age,"125,000.0000",48.4141,17.9010,18.0000,33.0000,48.0000,64.0000,79.0000
num_subscription_pauses,"125,000.0000",1.9911,1.4172,0.0000,1.0000,2.0000,3.0000,4.0000
weekly_hours,"125,000.0000",25.0370,14.4475,0.0001,12.4727,25.1167,37.5703,49.9999
average_session_length,"125,000.0000",1.0070,0.5731,0.0167,0.5107,1.0057,1.5039,1.9999
song_skip_rate,"125,000.0000",0.5008,0.2887,0.0000,0.2510,0.5012,0.7511,1.0000
weekly_songs_played,"125,000.0000",250.8239,143.3276,3.0000,127.0000,251.0000,375.0000,499.0000
weekly_unique_songs,"125,000.0000",121.2659,80.3908,3.0000,52.0000,110.0000,182.0000,299.0000
num_favorite_artists,"125,000.0000",24.4999,14.4460,0.0000,12.0000,25.0000,37.0000,49.0000
num_platform_friends,"125,000.0000",99.7132,57.6814,0.0000,50.0000,100.0000,150.0000,199.0000
num_playlists_created,"125,000.0000",49.4580,28.9353,0.0000,24.0000,49.0000,75.0000,99.0000



최종 데이터 결측치 요약


,missing_cnt,missing_ratio(%),nunique
weekly_hours,0,0.0000,125000
average_session_length,0,0.0000,124996
song_skip_rate,0,0.0000,124995
tenure_days,0,0.0000,2922
weekly_songs_played,0,0.0000,497
weekly_unique_songs,0,0.0000,297
num_platform_friends,0,0.0000,200
num_playlists_created,0,0.0000,100
age,0,0.0000,62
num_favorite_artists,0,0.0000,50



중복 행 수: 0

타깃(churned) 분포


,churned,count,ratio
0,1,64174,51.3400
1,0,60826,48.6600



상수 컬럼: []


In [13]:
# 수치형 이상치(IQR) 요약
# 연속형/실수형 중심 컬럼만 선택
iqr_cols = [
    'age', 'tenure_days', 'weekly_hours', 'average_session_length', 'song_skip_rate',
    'weekly_songs_played', 'weekly_unique_songs', 'num_favorite_artists',
    'num_platform_friends', 'num_playlists_created', 'num_shared_playlists',
    'notifications_clicked', 'State_TotalPop', 'State_AvgIncome'
]

outlier_rows = []
for col in iqr_cols:
    q1 = final_df[col].quantile(0.25)
    q3 = final_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    out_cnt = ((final_df[col] < lower) | (final_df[col] > upper)).sum()

    outlier_rows.append({
        'column': col,
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower': lower,
        'upper': upper,
        'outlier_cnt': int(out_cnt),
        'outlier_ratio(%)': round(out_cnt / len(final_df) * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values('outlier_ratio(%)', ascending=False)
display(outlier_summary.head(20))

# churned와의 상관계수 미리 보기
corr_with_target = (
    final_df.corr(numeric_only=True)['churned']
    .drop('churned')
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("\nchurned와의 상관계수 상위 변수")
display(corr_with_target.head(20).to_frame('corr_with_churned'))

,column,q1,q3,iqr,lower,upper,outlier_cnt,outlier_ratio(%)
12,State_TotalPop,"1,616,547.0000","9,845,333.0000","8,228,786.0000","-10,726,632.0000","22,188,512.0000",6665,5.3300
0,age,33.0000,64.0000,31.0000,-13.5000,110.5000,0,0.0000
2,weekly_hours,12.4727,37.5703,25.0977,-25.1738,75.2168,0,0.0000
3,average_session_length,0.5107,1.5039,0.9932,-0.9790,2.9937,0,0.0000
4,song_skip_rate,0.2510,0.7511,0.5001,-0.4992,1.5013,0,0.0000
1,tenure_days,728.0000,"2,190.0000","1,462.0000","-1,465.0000","4,383.0000",0,0.0000
5,weekly_songs_played,127.0000,375.0000,248.0000,-245.0000,747.0000,0,0.0000
6,weekly_unique_songs,52.0000,182.0000,130.0000,-143.0000,377.0000,0,0.0000
8,num_platform_friends,50.0000,150.0000,100.0000,-100.0000,300.0000,0,0.0000
7,num_favorite_artists,12.0000,37.0000,25.0000,-25.5000,74.5000,0,0.0000



churned와의 상관계수 상위 변수


,corr_with_churned
subscription_type_Free,0.3244
customer_service_inquiries_Low,-0.3183
weekly_hours,-0.3025
subscription_type_Premium,-0.2018
num_subscription_pauses,0.1830
song_skip_rate,0.1602
subscription_type_Student,0.0700
age,0.0487
notifications_clicked,-0.0424
weekly_unique_songs,0.0138


In [14]:
# eda_df
# - 원본 범주형이 살아 있는 데이터
# - 그룹별 boxplot, barplot, churn rate 비교용
eda_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125000 entries, 0 to 124999
Data columns (total 21 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   age                         125000 non-null  int64  
 1   location                    125000 non-null  str    
 2   subscription_type           125000 non-null  str    
 3   payment_plan                125000 non-null  str    
 4   num_subscription_pauses     125000 non-null  int64  
 5   payment_method              125000 non-null  str    
 6   customer_service_inquiries  125000 non-null  str    
 7   weekly_hours                125000 non-null  float64
 8   average_session_length      125000 non-null  float64
 9   song_skip_rate              125000 non-null  float64
 10  weekly_songs_played         125000 non-null  int64  
 11  weekly_unique_songs         125000 non-null  int64  
 12  num_favorite_artists        125000 non-null  int64  
 13  num_platform_friends     

In [15]:
# final_df
# - 인코딩 완료된 모델 입력용 데이터
# - 상관계수, 모델링, feature importance용
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125000 entries, 0 to 124999
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   age                                125000 non-null  int64  
 1   num_subscription_pauses            125000 non-null  int64  
 2   weekly_hours                       125000 non-null  float64
 3   average_session_length             125000 non-null  float64
 4   song_skip_rate                     125000 non-null  float64
 5   weekly_songs_played                125000 non-null  int64  
 6   weekly_unique_songs                125000 non-null  int64  
 7   num_favorite_artists               125000 non-null  int64  
 8   num_platform_friends               125000 non-null  int64  
 9   num_playlists_created              125000 non-null  int64  
 10  num_shared_playlists               125000 non-null  int64  
 11  notifications_clicked              125000 non-null